
# ViVQA Ablation Study cho Luận Văn
# Thực hiện Grid Search cho các Hyperparameters quan trọng để chứng minh tính khoa học.


In [1]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Điền thông tin Kaggle API của bạn vào đây
# (Vào Kaggle -> Settings -> Create New Token để lấy username và key)
os.environ['KAGGLE_USERNAME'] = "huyphngquc"
os.environ['KAGGLE_KEY'] = "d2051dddb134d7809d29527c977cab58"

# Tạo thư mục trên Drive để lưu dataset cố định (nếu chưa có)
DRIVE_DATASET_DIR = '/content/drive/MyDrive/Thesis_ViVQA_Data'
os.makedirs(DRIVE_DATASET_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# # ==============================================================================
# # CHỈ CHẠY 1 LẦN: Tải dataset từ Kaggle về thẳng Google Drive
# # SAU KHI TẢI XONG, HÃY COMMENT ĐOẠN CODE NÀY LẠI
# # ==============================================================================
# !kaggle datasets download -d hydrogenhydrogen/vivqa-1 -p {DRIVE_DATASET_DIR}

# print(f"✅ Đã tải file zip về: {DRIVE_DATASET_DIR}/vivqa-1.zip")

In [3]:
import os
import time

LOCAL_DATA_DIR = '/content/vivqa_dataset'
ZIP_PATH_ON_DRIVE = f'{DRIVE_DATASET_DIR}/vivqa-1.zip' # Đảm bảo biến này đúng
ZIP_PATH_LOCAL = '/content/vivqa-1.zip'

start_time = time.time()

# 1. Xóa rác cũ nếu có (Quan trọng để tránh giải nén đè lên file lỗi)
!rm -rf {LOCAL_DATA_DIR}
!rm -f {ZIP_PATH_LOCAL}

# 2. Copy bằng lệnh Linux (Trâu hơn shutil)
print("Đang copy từ Drive sang Local...")
!cp "{ZIP_PATH_ON_DRIVE}" "{ZIP_PATH_LOCAL}"

# 3. Giải nén và TỰ ĐỘNG SỬA LỖI (Dùng option -FF nếu cần)
print("Đang giải nén...")
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Thử giải nén bình thường trước
result = !unzip -q -n {ZIP_PATH_LOCAL} -d {LOCAL_DATA_DIR}

# Kiểm tra nếu vẫn còn báo lỗi "bad zipfile offset"
if any("bad zipfile offset" in line for line in result):
    print("⚠️ File zip có vẻ bị lỗi offset, đang thử fix...")
    # Lệnh fix file zip của linux
    !zip -F {ZIP_PATH_LOCAL} --out {ZIP_PATH_LOCAL}_fixed.zip
    !unzip -q -n {ZIP_PATH_LOCAL}_fixed.zip -d {LOCAL_DATA_DIR}

# 4. Dọn dẹp
if os.path.exists(ZIP_PATH_LOCAL): os.remove(ZIP_PATH_LOCAL)

print(f"✅ Xong! Kiểm tra thử: {len(os.listdir(LOCAL_DATA_DIR))} folders/files.")

✅ Xong! Kiểm tra thử: 13 folders/files.


In [4]:
# 4. Kiểm tra và in ra cấu trúc thực tế
print("\n--- KIỂM TRA CẤU TRÚC DATA ---")
if os.path.exists(LOCAL_DATA_DIR):
    # Liệt kê tất cả file và folder (kể cả trong folder con)
    all_files = []
    for root, dirs, files in os.walk(LOCAL_DATA_DIR):
        for name in files:
            all_files.append(os.path.join(root, name))
    
    print(f"📍 Path cơ sở: {LOCAL_DATA_DIR}")
    print(f"📦 Tổng số lượng file tìm thấy: {len(all_files)}")
    
    if len(all_files) > 0:
        print("📄 5 file đầu tiên để check path:")
        for f in all_files[:5]:
            print(f"   -> {f}")
    else:
        print("❌ CẢNH BÁO: Thư mục tồn tại nhưng không có file nào bên trong (do giải nén lỗi).")
else:
    print("❌ LỖI: Thư mục LOCAL_DATA_DIR không tồn tại.")


--- KIỂM TRA CẤU TRÚC DATA ---
📍 Path cơ sở: /content/vivqa_dataset
📦 Tổng số lượng file tìm thấy: 11720
📄 5 file đầu tiên để check path:
   -> /content/vivqa_dataset/merged_train.csv
   -> /content/vivqa_dataset/vqa_classified_answers.csv
   -> /content/vivqa_dataset/answer_space.txt
   -> /content/vivqa_dataset/answer_type_mapping.csv
   -> /content/vivqa_dataset/yolo_final_results.csv


In [5]:

# %%
!pip install -q transformers timm accelerate bitsandbytes scikit-learn underthesea pandas tqdm


In [6]:

import os
import ast
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import itertools
from tqdm.auto import tqdm
from transformers import Blip2Processor, Blip2Model, AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from underthesea import word_tokenize

import warnings
warnings.filterwarnings("ignore")

In [7]:
import warnings, logging
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("torch.utils.data.dataloader").setLevel(logging.ERROR)

In [ ]:


# ==========================================
# 1. ABLATION CONFIG (CẤU HÌNH LƯỚI THỬ NGHIỆM)
# ==========================================
CONFIG = {
    # Data paths (Sửa lại cho đúng môi trường Kaggle của bạn)
    "train_csv": "/content/vivqa_dataset/ViVQA-main/ViVQA-main/train.csv",
    "train_img_dir": "/content/vivqa_dataset/train",
    "train_box_csv": "/content/vivqa_dataset/merged_train.csv",
    "blip_model": "Salesforce/blip2-opt-2.7b",
    "text_model": "vinai/phobert-base",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "batch_size": 32, # Có thể giảm xuống 16 nếu bị OOM
    "type_embed_dim": 64,
    "max_length": 50,
    "box_required_types": [2],
    "qformer_unfreeze_layers": 2,
    
    # [PHASE 1] Cấu hình quét Training Loss
    # Dùng 1 tập con (Dev Set) để chạy nhanh
    "dev_set_ratio": 0.15, 
    "ablation_epochs": [0, 7, 20, 20], 
    "grid_training": {
        "lambda_bbox": [1.0, 2.0],
        "lambda_giou": [1.0, 5.0, 10.0]
    },
    
    # [PHASE 2] Cấu hình quét Inference
    "grid_inference": {
        "pad_ratio": [0.05, 0.15, 0.25],
        "crop_weight": [0.4, 0.6, 0.8]
    },
    
    # Nơi lưu kết quả để đưa vào luận văn
    "phase1_result_csv": "/content/drive/MyDrive/ablation_training_results.csv",
    "phase2_result_csv": "/content/drive/MyDrive/ablation_inference_results.csv",
    "phase3_result_csv": "/content/drive/MyDrive/ablation_structural_results.csv",
    "checkpoint_to_test": "/content/drive/MyDrive/vivqa_temp_best.pth" # Checkpoint sinh ra từ Phase 1 hoặc bạn tự trỏ vào
}

In [9]:
import sys

if not hasattr(sys, '_custom_unraisablehook_applied'):
    original_hook = sys.unraisablehook
    
    def filter_dataloader_del_errors(unraisable):
        # Nếu đúng là cái lỗi ngớ ngẩn của DataLoader thì im lặng bỏ qua
        if issubclass(unraisable.exc_type, AssertionError) and "can only test a child process" in str(unraisable.exc_value):
            pass
        else:
            # Nếu là lỗi khác thì vẫn in ra bình thường để còn debug
            original_hook(unraisable)

    sys.unraisablehook = filter_dataloader_del_errors
    sys._custom_unraisablehook_applied = True

In [10]:

# ==========================================
# 2. HÀM BỔ TRỢ
# ==========================================
def preprocess_vietnamese(text):
    text = str(text).lower().strip()
    return word_tokenize(text, format="text")

def find_image_path(folder, img_id):
    valid_exts = ['.jpg', '.png', '.jpeg']
    img_id_str = str(img_id)
    if img_id_str.lower().endswith(tuple(valid_exts)):
        path = os.path.join(folder, img_id_str)
        if os.path.exists(path): return path
    for ext in valid_exts:
        path = os.path.join(folder, f"{img_id_str}{ext}")
        if os.path.exists(path): return path
    return None

def calculate_iou(pred_boxes, target_boxes):
    # Sửa lỗi diện tích âm khi model đoán ngược tọa độ
    p_left = torch.min(pred_boxes[:, 0], pred_boxes[:, 2])
    p_top = torch.min(pred_boxes[:, 1], pred_boxes[:, 3])
    p_right = torch.max(pred_boxes[:, 0], pred_boxes[:, 2])
    p_bottom = torch.max(pred_boxes[:, 1], pred_boxes[:, 3])

    inter_xmin = torch.max(p_left, target_boxes[:, 0])
    inter_ymin = torch.max(p_top, target_boxes[:, 1])
    inter_xmax = torch.min(p_right, target_boxes[:, 2])
    inter_ymax = torch.min(p_bottom, target_boxes[:, 3])

    inter_w = torch.clamp(inter_xmax - inter_xmin, min=0)
    inter_h = torch.clamp(inter_ymax - inter_ymin, min=0)
    inter_area = inter_w * inter_h

    pred_area = (p_right - p_left) * (p_bottom - p_top)
    target_area = (target_boxes[:, 2] - target_boxes[:, 0]) * (target_boxes[:, 3] - target_boxes[:, 1])
    union_area = pred_area + target_area - inter_area + 1e-6

    iou = inter_area / union_area
    return torch.clamp(iou, min=0.0, max=1.0)


def giou_loss(pred_boxes, target_boxes):
    # Lấy tọa độ an toàn cho pred_boxes (tránh lỗi xmin > xmax sinh ra diện tích âm)
    px1, py1, px2, py2 = pred_boxes[:, 0], pred_boxes[:, 1], pred_boxes[:, 2], pred_boxes[:, 3]
    tx1, ty1, tx2, ty2 = target_boxes[:, 0], target_boxes[:, 1], target_boxes[:, 2], target_boxes[:, 3]

    p_left = torch.min(px1, px2)
    p_right = torch.max(px1, px2)
    p_top = torch.min(py1, py2)
    p_bottom = torch.max(py1, py2)

    pw = (p_right - p_left) + 1e-6
    ph = (p_bottom - p_top) + 1e-6
    pred_area = pw * ph

    target_area = (tx2 - tx1) * (ty2 - ty1)

    # Tính vùng giao nhau (Intersection)
    inter_xmin = torch.max(p_left, tx1)
    inter_ymin = torch.max(p_top, ty1)
    inter_xmax = torch.min(p_right, tx2)
    inter_ymax = torch.min(p_bottom, ty2)

    inter_w = torch.clamp(inter_xmax - inter_xmin, min=0)
    inter_h = torch.clamp(inter_ymax - inter_ymin, min=0)
    inter_area = inter_w * inter_h

    union_area = pred_area + target_area - inter_area + 1e-6
    iou = inter_area / union_area

    # Tính hộp bao quanh (Enclosing box)
    enc_xmin = torch.min(p_left, tx1)
    enc_ymin = torch.min(p_top, ty1)
    enc_xmax = torch.max(p_right, tx2)
    enc_ymax = torch.max(p_bottom, ty2)

    enc_w = torch.clamp(enc_xmax - enc_xmin, min=0)
    enc_h = torch.clamp(enc_ymax - enc_ymin, min=0)
    enc_area = enc_w * enc_h + 1e-6

    giou = iou - (enc_area - union_area) / enc_area
    return (1 - giou).mean()


In [11]:
# ==========================================
# 3. DATASET
# ==========================================
class ViVQATripleTaskDataset(Dataset):
    def __init__(self, dataframe, image_dir, blip_processor, tokenizer, label_encoder, unk_token_id):
        self.data = dataframe
        self.image_dir = image_dir
        self.blip_processor = blip_processor
        self.tokenizer = tokenizer
        self.label_encoder = label_encoder
        self.unk_token_id = unk_token_id
        self.known_classes = set(label_encoder.classes_)

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_id = row.get('img_id', row.get('image'))
        img_path = find_image_path(self.image_dir, img_id)
        try:
            image = Image.open(img_path).convert("RGB") if img_path else Image.new('RGB', (224, 224))
        except:
            image = Image.new('RGB', (224, 224))
        orig_w, orig_h = image.size
        pixel_values = self.blip_processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)

        target_box = torch.zeros(4, dtype=torch.float32)
        has_box = torch.tensor(0.0, dtype=torch.float32)
        
        # CẬP NHẬT: Ưu tiên lấy merged_box, nếu không có mới lấy owl_box
        box_str = row.get('merged_box', row.get('owl_box', None))
        
        if pd.notna(box_str):
            try:
                box = ast.literal_eval(str(box_str))
                xmin, ymin, xmax, ymax = box
                target_box[0], target_box[1] = xmin / orig_w, ymin / orig_h
                target_box[2], target_box[3] = xmax / orig_w, ymax / orig_h
                target_box = torch.clamp(target_box, 0.0, 1.0)
                has_box = torch.tensor(1.0, dtype=torch.float32)
            except:
                pass

        question = preprocess_vietnamese(row['question'])
        text_encoding = self.tokenizer(
            question, return_tensors="pt", padding="max_length",
            truncation=True, max_length=CONFIG['max_length'], add_special_tokens=True
        )
        answer = str(row.get('answer', '')).lower().strip()
        label = self.label_encoder.transform([answer])[0] if answer in self.known_classes else self.unk_token_id
        q_type = int(row.get('type', 0))

        return {
            "pixel_values": pixel_values,
            "input_ids": text_encoding['input_ids'].squeeze(0),
            "attention_mask": text_encoding['attention_mask'].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
            "q_type": torch.tensor(q_type, dtype=torch.long),
            "target_box": target_box,
            "has_box": has_box
        }

In [ ]:
# ==========================================
# 4. MODEL (CẬP NHẬT CHO STRUCTURAL ABLATION)
# ==========================================

class CrossAttentionFusion(nn.Module):
    def __init__(self, visual_dim, text_dim, embed_dim, num_heads=8):
        super().__init__()
        self.vis_project = nn.Linear(visual_dim, embed_dim)
        self.txt_project = nn.Linear(text_dim, embed_dim)
        self.multihead_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, dropout=0.1)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, visual_features, text_features):
        v_proj = self.vis_project(visual_features)
        t_proj = self.txt_project(text_features)
        attn_output, _ = self.multihead_attn(query=t_proj, key=v_proj, value=v_proj)
        combined = self.norm(attn_output + t_proj)
        return combined.mean(dim=1) + combined.max(dim=1)[0]


# --- THÊM MỚI: BBox Head cơ bản để làm Baseline ---
class BasicBBoxHead(nn.Module):
    """
    BBox Head truyền thống: Chỉ nhìn ảnh, không quan tâm Text hay Type.
    """
    def __init__(self, visual_dim=768):
        super().__init__()
        self.regressor = nn.Sequential(
            nn.Linear(visual_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 4),
            nn.Sigmoid()
        )

    def forward(self, vis_seq):
        # vis_seq shape: (B, 32, 768) -> Average Pooling -> (B, 768)
        vis_pooled = vis_seq.mean(dim=1)
        return self.regressor(vis_pooled)


class TextGuidedBBoxHead(nn.Module):
    
    def __init__(self, visual_dim=768, text_dim=768, type_dim=64):
        super().__init__()
        self.attn_proj = nn.Linear(text_dim + type_dim, visual_dim)
        self.vis_branch = nn.Sequential(
            nn.Linear(visual_dim, 512), nn.LayerNorm(512), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(512, 256), nn.ReLU(),
        )
        self.txt_branch = nn.Sequential(
            nn.Linear(text_dim + type_dim, 256), nn.LayerNorm(256), nn.ReLU(),
        )
        self.regressor = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 4), nn.Sigmoid()
        )

    def forward(self, vis_seq, text_feat, type_feat):
        context = torch.cat([text_feat, type_feat], dim=-1)         
        attn_query = self.attn_proj(context).unsqueeze(-1)           
        attn_weights = torch.bmm(vis_seq, attn_query).squeeze(-1)    
        attn_weights = torch.softmax(attn_weights, dim=-1).unsqueeze(-1)  
        attended_vis = (vis_seq * attn_weights).sum(dim=1)            

        vis_feat = self.vis_branch(attended_vis)   
        txt_feat = self.txt_branch(context)        
        combined = torch.cat([vis_feat, txt_feat], dim=-1)  
        return self.regressor(combined)


class ViVQATripleTaskModel(nn.Module):
    def __init__(self, num_classes, num_q_types, use_cross_attn=True, use_text_guided_box=True):
        super().__init__()
        # Cờ Ablation
        self.use_cross_attn = use_cross_attn
        self.use_text_guided_box = use_text_guided_box

        print(f"⏳ Khởi tạo Model | Cross-Attn: {use_cross_attn} | Text-Guided Box: {use_text_guided_box}")
        tmp_blip = Blip2Model.from_pretrained(CONFIG['blip_model'], torch_dtype=torch.float16)
        self.vision_model = tmp_blip.vision_model
        self.qformer = tmp_blip.qformer
        self.query_tokens = tmp_blip.query_tokens
        del tmp_blip

        for param in self.vision_model.parameters(): param.requires_grad = False
        for param in self.qformer.parameters(): param.requires_grad = False
        self.query_tokens.requires_grad = False
        self._unfreeze_qformer_last_layers(CONFIG['qformer_unfreeze_layers'])

        self.phobert = AutoModel.from_pretrained(CONFIG['text_model'])

        embed_dim = 768
        # Xử lý Ablation: Fusion
        if self.use_cross_attn:
            self.fusion = CrossAttentionFusion(visual_dim=768, text_dim=768, embed_dim=embed_dim)
        else:
            self.simple_fusion = nn.Linear(768 + 768, embed_dim) # Simple Concat
            self.fusion_norm = nn.LayerNorm(embed_dim)

        self.q_type_head = nn.Linear(768, num_q_types)
        self.type_emb = nn.Embedding(num_q_types, CONFIG['type_embed_dim'])

        in_features = embed_dim + CONFIG['type_embed_dim']
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512), nn.LayerNorm(512), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(512, num_classes)
        )
        
        # Xử lý Ablation: BBox Head
        if self.use_text_guided_box:
            self.bbox_head = TextGuidedBBoxHead(visual_dim=768, text_dim=768, type_dim=CONFIG['type_embed_dim'])
        else:
            self.bbox_head = BasicBBoxHead(visual_dim=768)

    def _unfreeze_qformer_last_layers(self, n_layers):
        try:
            layers = self.qformer.encoder.layer
            unfreeze_from = max(0, len(layers) - n_layers)
            for i, layer in enumerate(layers):
                if i >= unfreeze_from:
                    for param in layer.parameters(): param.requires_grad = True
            self.query_tokens.requires_grad = True
        except AttributeError:
            pass

    def forward(self, pixel_values, input_ids, attention_mask):
        with torch.no_grad():
            vision_outputs = self.vision_model(pixel_values=pixel_values)
            image_embeds = vision_outputs.last_hidden_state
            image_attention_mask = torch.ones(image_embeds.size()[:-1], dtype=torch.long, device=image_embeds.device)

        query_tokens = self.query_tokens.expand(image_embeds.shape[0], -1, -1)
        image_embeds_d = image_embeds.detach()
        query_outputs = self.qformer(
            query_embeds=query_tokens, encoder_hidden_states=image_embeds_d,
            encoder_attention_mask=image_attention_mask, output_attentions=False
        )
        visual_seq = query_outputs.last_hidden_state 

        text_outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        text_feats = text_outputs.last_hidden_state
        visual_seq = visual_seq.to(text_feats.dtype)

        cls_token_feats = text_feats[:, 0, :]
        q_type_logits = self.q_type_head(cls_token_feats)
        soft_q_type = torch.softmax(q_type_logits, dim=-1)
        type_vec = torch.matmul(soft_q_type, self.type_emb.weight)

        # Logic Ablation Fusion
        if self.use_cross_attn:
            fused_vec = self.fusion(visual_seq, text_feats)
        else:
            vis_pooled = visual_seq.mean(dim=1)
            concat_feats = torch.cat([vis_pooled, cls_token_feats], dim=-1)
            fused_vec = self.fusion_norm(self.simple_fusion(concat_feats))

        final_features = torch.cat([fused_vec, type_vec], dim=-1)
        ans_logits = self.classifier(final_features)

        # Logic Ablation BBox Head
        if self.use_text_guided_box:
            pred_boxes = self.bbox_head(visual_seq, cls_token_feats, type_vec)
        else:
            pred_boxes = self.bbox_head(visual_seq)

        return ans_logits, pred_boxes, q_type_logits

In [13]:

# ==========================================
# 3. CHUẨN BỊ DEV SET (TẬP DỮ LIỆU THU NHỎ)
# ==========================================
def prepare_dev_data():
    blip_processor = Blip2Processor.from_pretrained(CONFIG['blip_model'])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model'], use_fast=False)

    df_raw = pd.read_csv(CONFIG['train_csv'])
    if 'Unnamed: 0' in df_raw.columns: df_raw.rename(columns={'Unnamed: 0': 'question_id'}, inplace=True)
    df_box = pd.read_csv(CONFIG['train_box_csv'])
    if 'Unnamed: 0' in df_box.columns: df_box.rename(columns={'Unnamed: 0': 'question_id'}, inplace=True)

    df_merged = pd.merge(df_raw, df_box[['question_id', 'merged_box']], on='question_id', how='left')

    all_answers = df_merged['answer'].apply(lambda x: str(x).lower().strip()).unique().tolist()
    if 'unknown' not in all_answers: all_answers.append('unknown')
    label_encoder = LabelEncoder().fit(all_answers)
    unk_token_id = label_encoder.transform(['unknown'])[0]
    num_q_types = int(df_merged['type'].max() + 1)

    # Lấy 15% làm Dev Set, 5% làm Val set thu nhỏ để chạy thực nghiệm cho nhanh
    dev_df, val_df = train_test_split(df_merged, train_size=CONFIG['dev_set_ratio'], test_size=0.05, random_state=42, stratify=df_merged['type'])
    
    print(f"Kho dữ liệu Ablation | Train Dev: {len(dev_df)} mẫu | Val Dev: {len(val_df)} mẫu")

    dev_loader = DataLoader(ViVQATripleTaskDataset(dev_df, CONFIG['train_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id),
                            batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(ViVQATripleTaskDataset(val_df, CONFIG['train_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id),
                            batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

    return dev_loader, val_loader, label_encoder, num_q_types, blip_processor, tokenizer, unk_token_id, val_df



In [ ]:
# ==========================================
# 6. INFERENCE CHIẾN LƯỢC THEO Q-TYPE
# ==========================================
def crop_and_reencode(image_pil, box_coords, blip_processor, min_size=32, pad_ratio=0.15):
    W, H = image_pil.size
    xmin, ymin, xmax, ymax = box_coords

    pad_x = (xmax - xmin) * pad_ratio
    pad_y = (ymax - ymin) * pad_ratio
    xmin, ymin = max(0.0, xmin - pad_x), max(0.0, ymin - pad_y)
    xmax, ymax = min(1.0, xmax + pad_x), min(1.0, ymax + pad_y)

    px0, py0 = int(xmin * W), int(ymin * H)
    px1, py1 = int(xmax * W), int(ymax * H)

    if (px1 - px0) < min_size or (py1 - py0) < min_size:
        cropped = image_pil
    else:
        cropped = image_pil.crop((px0, py0, px1, py1))

    return blip_processor(images=cropped, return_tensors="pt")['pixel_values']


def multi_strategy_inference_batch(model, batch, blip_processor, device, box_required_types=None, crop_weight=0.6):
    if box_required_types is None: box_required_types = CONFIG['box_required_types']

    pixel_values = batch['pixel_values'].to(device, dtype=torch.float16)
    input_ids, attention_mask = batch['input_ids'].to(device), batch['attention_mask'].to(device)

    with torch.no_grad(), autocast():
        ans_logits_full, pred_boxes, q_type_logits = model(pixel_values, input_ids, attention_mask)

    pred_qtypes = q_type_logits.argmax(1)
    ans_logits_final = ans_logits_full.clone()

    box_indices = [i for i, qt in enumerate(pred_qtypes.tolist()) if qt in box_required_types]

    if box_indices and 'raw_images' in batch:
        for i in box_indices:
            raw_img = batch['raw_images'][i]
            
            # CẬP NHẬT: Model bắt buộc phải tự dùng box do nó dự đoán (No GT Data Leak)
            box_coords = pred_boxes[i].cpu().tolist()
            effective_crop_weight = crop_weight

            crop_pixels = crop_and_reencode(raw_img, box_coords, blip_processor).to(device, dtype=torch.float16)

            with torch.no_grad(), autocast():
                ans_logits_crop, _, _ = model(crop_pixels, input_ids[i:i+1], attention_mask[i:i+1])

            ans_logits_final[i] = ((1 - effective_crop_weight) * ans_logits_full[i] + effective_crop_weight * ans_logits_crop.squeeze(0))

    pred_answers = ans_logits_final.argmax(1)
    return pred_answers, pred_boxes, pred_qtypes

In [ ]:
import os
import gc
import torch
import itertools
import pandas as pd
from tqdm.auto import tqdm
from torch.cuda.amp import autocast, GradScaler

# ==========================================
# 4. PHASE 1: ABLATION STUDY CHO TRAINING LOSSES
# ==========================================
def run_phase1_training_ablation():
    print("\n" + "="*50)
    print("🚀 BẮT ĐẦU PHASE 1: TRAINING LOSS ABLATION")
    print("="*50)
    dev_loader, val_loader, label_encoder, num_q_types, _, _, _, _ = prepare_dev_data()
    
    # Tạo tổ hợp các parameter
    keys, values = zip(*CONFIG['grid_training'].items())
    experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    results = []
    best_score = 0
    
    # --- TỐI ƯU 1: KHỞI TẠO MODEL 1 LẦN DUY NHẤT ---
    print("⏳ Đang khởi tạo model gốc (chỉ 1 lần để tiết kiệm RAM)...")
    model = ViVQATripleTaskModel(len(label_encoder.classes_), num_q_types).to(CONFIG['device'])
    
    # Lưu trạng thái ban đầu để reset sau mỗi experiment
    init_weights_path = "/content/vivqa_init_weights_temp.pth"
    torch.save(model.state_dict(), init_weights_path)
    
    for exp_idx, exp_config in enumerate(experiments):
        print(f"\n>>> Experiment {exp_idx+1}/{len(experiments)}: {exp_config}")
        
        # --- TỐI ƯU 2: RESET MODEL VỀ TRẠNG THÁI GỐC ---
        model.load_state_dict(torch.load(init_weights_path))
        
        # Thiết lập lại optimizer
        other_params = [p for n, p in model.named_parameters() if p.requires_grad and 'qformer' not in n and n != 'query_tokens']
        qformer_params = [p for p in model.qformer.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW([{'params': other_params, 'lr': 5e-5}, 
                                       {'params': qformer_params + [model.query_tokens], 'lr': 1e-5}])
        
        criterion_vqa = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
        criterion_qtype = torch.nn.CrossEntropyLoss()
        criterion_bbox = torch.nn.SmoothL1Loss(reduction='none')
        scaler = GradScaler()
        
        # Train
        for epoch in range(CONFIG['ablation_epochs'][1]):
            model.train()
            for batch in tqdm(dev_loader, desc=f"Train Ep {epoch+1}", leave=False):
                pixel_values = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
                input_ids = batch['input_ids'].to(CONFIG['device'])
                attention_mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                q_types_gt = batch['q_type'].to(CONFIG['device'])
                target_box = batch['target_box'].to(CONFIG['device'])
                has_box = batch['has_box'].to(CONFIG['device'])

                optimizer.zero_grad()
                with autocast():
                    ans_logits, pred_boxes, q_type_logits = model(pixel_values, input_ids, attention_mask)
                    loss_vqa = criterion_vqa(ans_logits, labels)
                    loss_qtype = criterion_qtype(q_type_logits, q_types_gt)

                    target_types = torch.tensor(CONFIG['box_required_types'], device=CONFIG['device'])
                    color_mask = torch.isin(q_types_gt, target_types).float()
                    valid_box_mask = (has_box == 1.0) & (color_mask == 1.0)

                    loss_bbox_raw = criterion_bbox(pred_boxes, target_box).mean(dim=1)
                    loss_bbox = (loss_bbox_raw * valid_box_mask).sum() / (valid_box_mask.sum() + 1e-6)
                    
                    if valid_box_mask.any():
                        loss_giou = giou_loss(pred_boxes[valid_box_mask].float(), target_box[valid_box_mask].float()) 
                    else:
                        loss_giou = torch.tensor(0.0, device=CONFIG['device'])

                    # ÁP DỤNG TRỌNG SỐ TỪ GRID SEARCH
                    loss = loss_vqa + loss_qtype + (exp_config['lambda_bbox'] * loss_bbox) + (exp_config['lambda_giou'] * loss_giou)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        # Đánh giá trên Val Dev
        model.eval()
        val_vqa_c, val_total, iou_sum, iou_count = 0, 0, 0.0, 0
        with torch.no_grad(), autocast():
            for batch in val_loader:
                pixel_values = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
                input_ids = batch['input_ids'].to(CONFIG['device'])
                attention_mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                target_box = batch['target_box'].to(CONFIG['device'])
                has_box = batch['has_box'].to(CONFIG['device'])
                
                ans_logits, pred_boxes, _ = model(pixel_values, input_ids, attention_mask)
                val_vqa_c += (ans_logits.argmax(1) == labels).sum().item()
                val_total += labels.size(0)
                
                valid_box_mask = (has_box == 1.0)
                if valid_box_mask.any():
                    ious = calculate_iou(pred_boxes[valid_box_mask], target_box[valid_box_mask])
                    iou_sum += ious.sum().item()
                    iou_count += ious.size(0)

        val_acc = val_vqa_c / val_total
        val_miou = (iou_sum / iou_count) if iou_count > 0 else 0.0
        combined_score = (val_acc + val_miou * 1.5) / 2.5
        
        print(f"-> Kết quả: VQA Acc: {val_acc:.4f} | mIoU: {val_miou:.4f} | Score: {combined_score:.4f}")
        
        # Lưu best checkpoint để xài cho phase 2
        if combined_score > best_score:
            best_score = combined_score
            torch.save(model.state_dict(), CONFIG['checkpoint_to_test'])
        
        exp_config['val_vqa_acc'] = val_acc
        exp_config['val_miou'] = val_miou
        exp_config['score'] = combined_score
        results.append(exp_config)
        
        # --- TỐI ƯU 3: DỌN RÁC BỘ NHỚ SAU MỖI EXPERIMENT ---
        del optimizer
        gc.collect()
        torch.cuda.empty_cache()
        
    # Xóa file weights tạm sau khi chạy xong tất cả
    if os.path.exists(init_weights_path):
        os.remove(init_weights_path)
        
    df_results = pd.DataFrame(results)
    df_results.to_csv(CONFIG['phase1_result_csv'], index=False)
    print(f"✅ Đã lưu kết quả Phase 1 tại: {CONFIG['phase1_result_csv']}")
    return df_results


# ==========================================
# 5. PHASE 2: ABLATION STUDY CHO INFERENCE
# ==========================================
def run_phase2_inference_ablation():
    print("\n" + "="*50)
    print("🎯 BẮT ĐẦU PHASE 2: INFERENCE ABLATION (PAD & CROP)")
    print("="*50)
    
    _, _, label_encoder, num_q_types, blip_processor, tokenizer, unk_token_id, val_df = prepare_dev_data()
    
    # Dataset đặc biệt cho Inference phải có raw_images
    class ViVQAInferenceAblation(ViVQATripleTaskDataset):
        def __getitem__(self, idx):
            item = super().__getitem__(idx)
            img_id = self.data.iloc[idx].get('img_id', self.data.iloc[idx].get('image'))
            img_path = find_image_path(self.image_dir, img_id)
            item['raw_image'] = Image.open(img_path).convert("RGB") if img_path else Image.new('RGB', (224, 224))
            return item

    def collate_fn(batch):
        raw_images = [item.pop('raw_image') for item in batch]
        collated = torch.utils.data.default_collate(batch)      
        collated['raw_images'] = raw_images
        return collated

    infer_loader = torch.utils.data.DataLoader(
        ViVQAInferenceAblation(val_df, CONFIG['train_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id),
        batch_size=16, shuffle=False, num_workers=2, collate_fn=collate_fn
    )

    model = ViVQATripleTaskModel(len(label_encoder.classes_), num_q_types).to(CONFIG['device'])
    model.load_state_dict(torch.load(CONFIG['checkpoint_to_test'], map_location=CONFIG['device']))
    model.eval()

    keys, values = zip(*CONFIG['grid_inference'].items())
    experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]
    results = []

    for exp_idx, exp_config in enumerate(experiments):
        print(f"\n>>> Thử nghiệm {exp_idx+1}/{len(experiments)}: {exp_config}")
        
        val_vqa_c, color_q_correct, color_q_total, val_total = 0, 0, 0, 0
        
        with torch.no_grad():
            for batch in tqdm(infer_loader, leave=False):
                labels = batch['labels'].to(CONFIG['device'])
                q_types_gt = batch['q_type'].to(CONFIG['device'])
                
                # Đảm bảo hàm multi_strategy_inference_batch nhận tham số pad_ratio và crop_weight
                pred_answers, pred_boxes, pred_qtypes = multi_strategy_inference_batch(
                    model, batch, blip_processor, CONFIG['device'],
                    box_required_types=CONFIG['box_required_types'],
                    crop_weight=exp_config.get('crop_weight', 0.5),
                    pad_ratio=exp_config.get('pad_ratio', 0.1) # Truyền pad_ratio vào đây
                )

                val_vqa_c += (pred_answers == labels).sum().item()
                val_total += labels.size(0)
                
                # Tính độ chính xác riêng cho Type 2 (Màu sắc)
                for p_ans, t_ans, qt in zip(pred_answers, labels, q_types_gt):
                    if qt.item() in CONFIG['box_required_types']:
                        color_q_total += 1
                        if p_ans.item() == t_ans.item():
                            color_q_correct += 1

        acc_overall = val_vqa_c / val_total
        acc_color = color_q_correct / color_q_total if color_q_total > 0 else 0
        
        print(f"-> VQA Overall: {acc_overall:.4f} | VQA Color Type (Ảnh hưởng bởi Box): {acc_color:.4f}")
        
        exp_config['acc_overall'] = acc_overall
        exp_config['acc_color_type'] = acc_color
        results.append(exp_config)

        # --- TỐI ƯU 4: DỌN RÁC BỘ NHỚ SAU INFERENCE CACHE ---
        gc.collect()
        torch.cuda.empty_cache()

    df_results = pd.DataFrame(results)
    df_results.to_csv(CONFIG['phase2_result_csv'], index=False)
    print(f"✅ Đã lưu kết quả Phase 2 tại: {CONFIG['phase2_result_csv']}")
    return df_results




🚀 BẮT ĐẦU PHASE 1: TRAINING LOSS ABLATION


The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Kho dữ liệu Ablation | Train Dev: 1799 mẫu | Val Dev: 600 mẫu
⏳ Đang khởi tạo model gốc (chỉ 1 lần để tiết kiệm RAM)...
⏳ Đang load BLIP-2 Vision & Q-Former...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



>>> Thử nghiệm 1/9: {'pad_ratio': 0.05, 'crop_weight': 0.4}


  0%|          | 0/38 [00:00<?, ?it/s]

NameError: name 'multi_strategy_inference_batch' is not defined

In [ ]:
# ==========================================
# 6. PHASE 3: STRUCTURAL ABLATION (KIẾN TRÚC)
# ==========================================
def run_phase3_structural_ablation():
    print("\n" + "="*60)
    print("🏗️ BẮT ĐẦU PHASE 3: STRUCTURAL ABLATION (TẮT/BẬT MODULE)")
    print("="*60)
    
    dev_loader, val_loader, label_encoder, num_q_types, _, _, _, _ = prepare_dev_data()
    
    # Định nghĩa 4 trạng thái tiến hóa của mô hình
    structural_configs = [
        {
            "name": "1. Baseline (VQA Only, Simple Concat)",
            "use_cross_attn": False, "use_text_guided_box": False,
            "lambda_bbox": 0.0, "lambda_qtype": 0.0, "lambda_giou": 0.0
        },
        {
            "name": "2. + Multi-task (Basic Box Head)",
            "use_cross_attn": False, "use_text_guided_box": False,
            "lambda_bbox": 2.0, "lambda_qtype": 1.0, "lambda_giou": 10.0 # Dùng lambda tốt nhất từ Phase 1
        },
        {
            "name": "3. + CrossAttentionFusion",
            "use_cross_attn": True, "use_text_guided_box": False,
            "lambda_bbox": 2.0, "lambda_qtype": 1.0, "lambda_giou": 10.0
        },
        {
            "name": "4. + TextGuidedBox (Proposed Full Model)",
            "use_cross_attn": True, "use_text_guided_box": True,
            "lambda_bbox": 2.0, "lambda_qtype": 1.0, "lambda_giou": 10.0
        }
    ]
    
    results = []

    for exp_config in structural_configs:
        print(f"\n>>> Đang train cấu hình: {exp_config['name']}")
        
        # Khởi tạo model theo cờ Ablation
        model = ViVQATripleTaskModel(
            num_classes=len(label_encoder.classes_), 
            num_q_types=num_q_types,
            use_cross_attn=exp_config['use_cross_attn'],
            use_text_guided_box=exp_config['use_text_guided_box']
        ).to(CONFIG['device'])
        
        other_params = [p for n, p in model.named_parameters() if p.requires_grad and 'qformer' not in n and n != 'query_tokens']
        qformer_params = [p for p in model.qformer.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW([{'params': other_params, 'lr': 5e-5}, 
                                       {'params': qformer_params + [model.query_tokens], 'lr': 1e-5}])
        
        criterion_vqa = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
        criterion_qtype = torch.nn.CrossEntropyLoss()
        criterion_bbox = torch.nn.SmoothL1Loss(reduction='none')
        scaler = GradScaler()
        
        # Train nhanh trong số epoch định sẵn (vd: 3 epochs)
        for epoch in range(CONFIG['ablation_epochs'][3]
            model.train()
            for batch in tqdm(dev_loader, desc=f"Train Ep {epoch+1}", leave=False):
                pixel_values = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
                input_ids = batch['input_ids'].to(CONFIG['device'])
                attention_mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                q_types_gt = batch['q_type'].to(CONFIG['device'])
                target_box = batch['target_box'].to(CONFIG['device'])
                has_box = batch['has_box'].to(CONFIG['device'])

                optimizer.zero_grad()
                with autocast():
                    ans_logits, pred_boxes, q_type_logits = model(pixel_values, input_ids, attention_mask)
                    loss_vqa = criterion_vqa(ans_logits, labels)
                    
                    loss = loss_vqa
                    
                    # Chỉ cộng thêm loss nếu cờ Multi-task được bật
                    if exp_config['lambda_bbox'] > 0:
                        loss_qtype = criterion_qtype(q_type_logits, q_types_gt)
                        target_types = torch.tensor(CONFIG['box_required_types'], device=CONFIG['device'])
                        color_mask = torch.isin(q_types_gt, target_types).float()
                        valid_box_mask = (has_box == 1.0) & (color_mask == 1.0)

                        loss_bbox_raw = criterion_bbox(pred_boxes, target_box).mean(dim=1)
                        loss_bbox = (loss_bbox_raw * valid_box_mask).sum() / (valid_box_mask.sum() + 1e-6)
                        loss_giou = giou_loss(pred_boxes[valid_box_mask].float(), target_box[valid_box_mask].float()) if valid_box_mask.any() else torch.tensor(0.0, device=CONFIG['device'])

                        loss = loss + (exp_config['lambda_qtype'] * loss_qtype) + (exp_config['lambda_bbox'] * loss_bbox) + (exp_config['lambda_giou'] * loss_giou)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        # Đánh giá trên Val Dev
        model.eval()
        val_vqa_c, val_total, iou_sum, iou_count = 0, 0, 0.0, 0
        with torch.no_grad(), autocast():
            for batch in val_loader:
                pixel_values = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
                input_ids = batch['input_ids'].to(CONFIG['device'])
                attention_mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                target_box = batch['target_box'].to(CONFIG['device'])
                has_box = batch['has_box'].to(CONFIG['device'])
                
                ans_logits, pred_boxes, _ = model(pixel_values, input_ids, attention_mask)
                val_vqa_c += (ans_logits.argmax(1) == labels).sum().item()
                val_total += labels.size(0)
                
                valid_box_mask = (has_box == 1.0)
                if valid_box_mask.any() and exp_config['lambda_bbox'] > 0:
                    ious = calculate_iou(pred_boxes[valid_box_mask], target_box[valid_box_mask])
                    iou_sum += ious.sum().item(); iou_count += ious.size(0)

        val_acc = val_vqa_c / val_total
        val_miou = (iou_sum / iou_count) if iou_count > 0 else 0.0
        
        print(f"-> KQ {exp_config['name']}: VQA Acc: {val_acc:.4f} | mIoU: {val_miou:.4f}")
        
        results.append({
            "Model Configuration": exp_config['name'],
            "VQA Accuracy": val_acc,
            "mIoU": val_miou if exp_config['lambda_bbox'] > 0 else "N/A"
        })
        
        # Dọn rác
        del model, optimizer
        import gc; gc.collect(); torch.cuda.empty_cache()
        
    df_results = pd.DataFrame(results)
    df_results.to_csv(CONFIG['phase3_result_csv'], index=False)
    print(f"✅ Đã lưu kết quả Phase 3 tại: {CONFIG['phase3_result_csv']}")
    return df_results

In [ ]:

# 1. Chạy Phase 1: Tìm Trọng số (Bạn đã chạy rồi thì có thể comment lại)
df_phase1 = run_phase1_training_ablation()
print("\n[BẢNG 1] Thông số Training tối ưu:")
print(df_phase1.sort_values(by='score', ascending=False).to_markdown())


In [ ]:

# 2. Chạy Phase 2: Chiến lược Inference
df_phase2 = run_phase2_inference_ablation()
print("\n[BẢNG 2] Thông số Inference tối ưu:")
print(df_phase2.sort_values(by='acc_color_type', ascending=False).to_markdown())


In [ ]:

# 3. Chạy Phase 3: Bảng Vàng (Structural Ablation)
df_phase3 = run_phase3_structural_ablation()
print("\n🏆 [BẢNG 3] CHỨNG MINH KIẾN TRÚC RANK B (STRUCTURAL ABLATION):")
print(df_phase3.to_markdown(index=False))